# Doppler ultrasound: measuring blood flow

*medical · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/doppler_ultrasound)

An ultrasound probe listens to the echoes of red blood cells. Each moving cell shifts the echo by $f_D = 2 f_0 v \cos\theta / c$, and hundreds of cells in a pulsing artery are simulated at once as one batch of phasors. A short-time FFT turns their summed echo into the Doppler spectrogram a scanner shows: the spread of blood velocities across the cardiac cycle.

**Vector form**

$$f_D = \frac{2 f_0\, \mathbf v\cdot\hat{\mathbf k}}{c} = \frac{2 f_0\, v\cos\theta}{c},\qquad v(r, t) = v_{\max}(t)\Big(1 - \frac{r^2}{R^2}\Big)$$

**Index (tensor) form**

$$v_{tn} = v_{\max,t}\,\big(1 - r_n^2/R^2\big),\qquad s_t = a_n\, e^{i\phi_{tn}},\quad \phi_{tn} = \phi_n + 2\pi \textstyle\sum_{t'\le t} f_{D,t'n}\,\Delta t,\qquad P_{fk} = \big|\textstyle\sum_t w_{t-kH}\, s_t\, e^{-2\pi i f t}\big|^2$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $v_{\max,t}$ | `cardiac_waveform(t, v_peak, heart_rate)` | (T,) | centreline velocity over two heartbeats: systolic peak, brief backflow, slow diastolic flow |
| $r_n,\ \phi_n,\ a_n$ | `scatterers(n, R, length, dev)` | (N,) | random red blood cells: radius in the vessel, echo phase and strength |
| $v_{tn}$ | `einsum('t,n->tn', v_max, profile)` | (T, N) | parabolic (laminar) profile times the waveform: an outer product, no index summed |
| $f_{D,tn}$ | `doppler_shift(v, f0, theta)` | (T, N) | Doppler shift $2 f_0 v \cos\theta / c$ of every cell at every pulse |
| $s_t$ | `einsum('n,tn->t', a, exp(1j*phi))` | (T,) | repeated $n$ ⇒ summed: the received echo, one complex (IQ) sample per pulse at $\Delta t = 1/\mathrm{PRF}$ |
| $P_{fk}$ | `torch.stft(s, n_fft, hop, window)` | (F, K) | spectrogram: power at frequency $f$ in time window $k$ |
| $\tilde f$ | `alias(f, prf)` | (…) | pulsed Doppler only sees $f$ wrapped into $[-\mathrm{PRF}/2, \mathrm{PRF}/2)$ |

The factor 2 appears because the shift happens twice: the moving cell receives a shifted wave and re-emits it while moving. Only the velocity component along the beam counts, so at $\theta = 90°$ there is no shift at all; scanners ask the operator for the angle and compute $v = f_D c / (2 f_0 \cos\theta)$, which becomes very uncertain above about 60°. With the defaults ($f_0 = 5$ MHz, $\theta = 60°$, peak 1 m/s) the systolic shift is about 3.2 kHz, in the audible range, which is why Doppler machines also play the signal as sound. In laminar flow the profile is parabolic: cells near the wall barely move and the centreline moves fastest. Because equal areas of the cross-section hold equal numbers of cells, every speed between 0 and $v_{\max}$ is equally common, the spectrum is a flat band ("filled-in" window), and the mean frequency is exactly half the maximum. Disturbed or stenotic flow would broaden the spectrum instead. Pulsed Doppler samples the echo once per pulse, so by Nyquist it can only measure $|f_D| < \mathrm{PRF}/2$: lower the PRF (or raise the velocity) and the systolic peak wraps around to the negative side (aliasing). Real scanners need a high PRF for fast jets but a low PRF to reach deep vessels, since each pulse must return before the next is sent. Simplifications: the sample volume covers the whole vessel, the beam is uniform, the IQ signal is simulated directly at baseband, and there is no wall filter or tissue clutter. All values are approximate and for education only, not for clinical use.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/doppler_ultrasound/main.py
"""Main loop: parameters → tensors → simulate the echo signal → spectrogram → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors

R_VESSEL = 3e-3  # m, artery radius
LENGTH_MM = 20.0  # length of vessel shown in the animation


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    f0 = params["f0"] * 1e6  # MHz → Hz
    prf = params["prf"] * 1e3  # kHz → Hz
    theta = math.radians(params["theta"])

    t = tensors.sample_times(prf, params["heart_rate"], 2, dev)  # (T,)
    v_max = tensors.cardiac_waveform(t, params["v_peak"], params["heart_rate"])  # (T,)
    scat = tensors.scatterers(params["n_scatterers"], R_VESSEL, LENGTH_MM, dev)

    v, s = solve.simulate(v_max, scat, R_VESSEL, f0, theta, prf)
    s = s + tensors.noise(len(s), 0.03 * math.sqrt(params["n_scatterers"]), dev)
    win_t, freqs, power = solve.spectrogram(s, prf)

    f_peak = equation.doppler_shift(v_max.max(), f0, theta).item()
    v_nyq = abs(prf / 2 * equation.C_TISSUE / (2 * f0 * math.cos(theta))) if math.cos(theta) > 1e-6 else math.inf
    print(f"{len(t)} pulses at PRF {prf / 1e3:.1f} kHz × {len(scat['r'])} scatterers → signal {tuple(s.shape)}")
    print(f"peak systolic velocity {v_max.max().item():.2f} m/s → Doppler shift {f_peak / 1e3:.2f} kHz")
    print(f"Nyquist limit PRF/2 = {prf / 2e3:.2f} kHz ↔ {v_nyq:.2f} m/s along the vessel")
    print("aliasing: peak wraps to the other side of the spectrum" if f_peak > prf / 2 else "no aliasing")
    return plot.animate(t, v, v_max, scat, win_t, freqs, power, f0, theta, prf, LENGTH_MM, out_dir / "doppler.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/doppler_ultrasound/equation.py
r"""Physics → tensor translation.

An ultrasound wave of frequency f_0 hits a red blood cell moving at speed v. The cell first receives a
shifted frequency (moving observer) and then re-emits it (moving source), so the echo is shifted twice:

    f_D = \frac{2 f_0 v \cos\theta}{c},      θ = angle between the beam and the flow,  c ≈ 1540 m/s in tissue.

In a vessel with laminar (Poiseuille) flow every scatterer n at radius r_n moves with

    v_{tn} = v_{\max}(t) \, (1 - r_n^2 / R^2)          (outer product: no index summed)

and the received (demodulated, complex IQ) signal is the sum of one rotating phasor per scatterer:

    s_t = a_n \, e^{i \phi_{tn}},     \phi_{tn} = \phi_n + 2\pi \sum_{t' \le t} f_{D,t'n} \, \Delta t

The repeated index n is summed — that is einsum('n,tn->t').  The real part of s_t is a sum of cosines
with random phases φ_n.
"""

import math

import torch

C_TISSUE = 1540.0  # m/s, average speed of sound in soft tissue


def doppler_shift(v: torch.Tensor, f0: float, theta: float, c: float = C_TISSUE) -> torch.Tensor:
    r"""f_D = 2 f_0 v \cos\theta / c   (any shape, Hz).   θ in radians."""
    return 2.0 * f0 * v * math.cos(theta) / c


def velocity_field(v_max: torch.Tensor, r: torch.Tensor, R: float) -> torch.Tensor:
    r"""v_{tn} = v_max(t) (1 - r_n²/R²).   v_max: (T,), r: (N,) → (T, N)"""
    profile = 1.0 - (r / R) ** 2  # (N,) parabolic profile: 1 on the axis, 0 at the wall
    return torch.einsum("t,n->tn", v_max, profile)


def received_signal(amplitude: torch.Tensor, phase0: torch.Tensor, f_D: torch.Tensor, dt: float) -> torch.Tensor:
    r"""s_t = a_n exp(i φ_{tn}),  φ_{tn} = φ_n + 2π Σ_{t'≤t} f_{D,t'n} Δt.   f_D: (T, N) → s: (T,) complex.

    The phase is the running integral of the frequency, so a scatterer whose speed changes over the cardiac
    cycle keeps a continuous phase.  Sampling it every Δt = 1/PRF is what makes pulsed Doppler alias.
    """
    phase = phase0 + 2 * math.pi * torch.cumsum(f_D, dim=0) * dt  # (T, N)
    phasors = torch.polar(torch.ones_like(phase), phase)  # (T, N) complex: e^{iφ}
    return torch.einsum("n,tn->t", amplitude.to(phasors.dtype), phasors)  # repeated n ⇒ summed


def alias(f: torch.Tensor, prf: float) -> torch.Tensor:
    """Frequency seen after sampling at the pulse repetition frequency: wrapped into [-PRF/2, PRF/2)."""
    return torch.remainder(f + prf / 2, prf) - prf / 2


def velocity_from_frequency(f: torch.Tensor, f0: float, theta: float, c: float = C_TISSUE) -> torch.Tensor:
    r"""Invert the Doppler equation: v = f_D c / (2 f_0 \cos\theta)."""
    return f * c / (2.0 * f0 * math.cos(theta))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/doppler_ultrasound/tensors.py
"""Build the concrete tensors: sample times, the pulsatile centreline velocity and the random scatterers."""

import math

import torch


def sample_times(prf: float, heart_rate: float, n_beats: int, device: torch.device) -> torch.Tensor:
    """Slow-time samples t, one per transmitted pulse (Δt = 1/PRF) over n_beats cardiac cycles: shape (T,)."""
    duration = n_beats * 60.0 / heart_rate
    n = int(duration * prf)
    return torch.arange(n, device=device) / prf


def cardiac_waveform(t: torch.Tensor, v_peak: float, heart_rate: float) -> torch.Tensor:
    """Centreline velocity v_max(t) of an artery, shape (T,): a sharp systolic peak, a short reverse-flow dip
    and slow forward flow in diastole (a stylised triphasic waveform, roughly like a leg artery)."""
    period = 60.0 / heart_rate
    tau = torch.remainder(t, period)  # time since the start of the beat
    systole = torch.exp(-(((tau - 0.12) / 0.045) ** 2))  # peak 0.12 s after the beat starts
    reverse = torch.exp(-(((tau - 0.27) / 0.04) ** 2))  # brief backflow as the valve closes
    return v_peak * (0.2 + 0.8 * systole - 0.35 * reverse)


def scatterers(n: int, R: float, length: float, device: torch.device, seed: int = 0) -> dict[str, torch.Tensor]:
    """n red-blood-cell scatterers spread uniformly over the vessel cross-section, all shape (N,).

    r       radial distance from the axis (r = R √u gives a uniform density over the disc)
    phase   random starting phase φ_n of each echo
    amp     echo amplitude a_n (cells scatter with random strength)
    x, y    position in the longitudinal view used by the animation (y = r cos α is the projection)
    """
    g = torch.Generator().manual_seed(seed)
    r = R * torch.sqrt(torch.rand(n, generator=g))
    alpha = 2 * math.pi * torch.rand(n, generator=g)
    out = {
        "r": r,
        "phase": 2 * math.pi * torch.rand(n, generator=g),
        "amp": 0.5 + torch.rand(n, generator=g),
        "x": length * torch.rand(n, generator=g),
        "y": r * torch.cos(alpha),
    }
    return {k: v.to(device) for k, v in out.items()}


def noise(n: int, sigma: float, device: torch.device, seed: int = 1) -> torch.Tensor:
    """Complex white receiver noise of standard deviation sigma per sample: shape (T,)."""
    g = torch.Generator().manual_seed(seed)
    return (sigma / math.sqrt(2) * torch.complex(torch.randn(n, generator=g), torch.randn(n, generator=g))).to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/doppler_ultrasound/solve.py
"""Simulate the Doppler signal of all scatterers and turn it into a spectrogram with a short-time FFT."""

import torch

from .equation import doppler_shift, received_signal, velocity_field


def simulate(
    v_max: torch.Tensor, scat: dict[str, torch.Tensor], R: float, f0: float, theta: float, prf: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (velocities v_{tn} (T, N), received IQ signal s_t (T,) complex), sampled once per pulse."""
    v = velocity_field(v_max, scat["r"], R)  # (T, N)
    f_D = doppler_shift(v, f0, theta)  # (T, N) true (unaliased) shift of every scatterer
    s = received_signal(scat["amp"], scat["phase"], f_D, 1.0 / prf)  # (T,)
    return v, s


def spectrogram(
    s: torch.Tensor, prf: float, n_fft: int = 128, hop: int = 32
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Short-time Fourier transform of the complex signal.

    Returns (window times (K,), frequencies (F,) from -PRF/2 to +PRF/2, power (F, K)).
    A complex signal has a two-sided spectrum: positive frequencies = flow towards the transducer.
    """
    window = torch.hann_window(n_fft, device=s.device)
    S = torch.stft(s, n_fft, hop_length=hop, window=window, center=True, return_complex=True, onesided=False)
    power = torch.fft.fftshift(S.abs() ** 2, dim=0) / window.pow(2).sum()  # (F, K), zero frequency in the middle
    freqs = torch.fft.fftshift(torch.fft.fftfreq(n_fft, d=1.0 / prf, device=s.device))  # (F,)
    times = torch.arange(power.shape[1], device=s.device) * hop / prf  # (K,) window centres
    return times, freqs, power


def mean_frequency(freqs: torch.Tensor, power: torch.Tensor) -> torch.Tensor:
    """Power-weighted mean frequency of each spectrum (the 'mean velocity' trace of a scanner): (K,)."""
    return torch.einsum("f,fk->k", freqs, power) / power.sum(0)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/doppler_ultrasound/plot.py
"""Animate the vessel with moving scatterers and the transducer beam, next to the spectrogram as it builds up."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Arc

from problems._common.plotting import frame_indices, save_gif

from .equation import doppler_shift

SLOW = 0.1  # the animation shows the cells moving 10× slower than real time


def animate(
    t: torch.Tensor,
    v: torch.Tensor,
    v_max: torch.Tensor,
    scat: dict[str, torch.Tensor],
    win_t: torch.Tensor,
    freqs: torch.Tensor,
    power: torch.Tensor,
    f0: float,
    theta: float,
    prf: float,
    length: float,
    path: Path,
) -> Path:
    dt = 1.0 / prf
    times = t.cpu().numpy()
    frames = frame_indices(len(times), 75)
    # displacement of every cell (mm) at the rendered frames only: running integral of its velocity
    x_mm = (scat["x"] + SLOW * 1e3 * torch.cumsum(v, dim=0) * dt)[frames]  # (frames, N)
    x_mm = torch.remainder(x_mm, length).cpu().numpy()
    speed = v[frames].abs().cpu().numpy()  # (frames, N)
    y_mm = (scat["y"] * 1e3).cpu().numpy()
    R_mm = float(scat["r"].max()) * 1e3 * 1.02

    fig, (ax, ax_s) = plt.subplots(1, 2, figsize=(9, 3.8), gridspec_kw={"width_ratios": [1, 1.45]})

    # --- left: longitudinal view of the vessel, transducer and beam -----------------------------------
    ax.set_xlim(-1, length + 1)
    ax.set_ylim(-R_mm - 1.5, 13)
    ax.set_aspect("equal")
    ax.set_xlabel("along vessel (mm)")
    ax.set_ylabel("mm")
    ax.grid(False)
    ax.fill_between([-1, length + 1], -R_mm, R_mm, color="#fee2e2", zorder=0)
    for y in (-R_mm, R_mm):
        ax.axhline(y, color="#374151", lw=2)
    centre = np.array([length / 2, 0.0])
    beam = np.array([math.cos(theta), math.sin(theta)])  # from the vessel centre towards the transducer
    probe = centre + 10.5 * beam
    side = np.array([-beam[1], beam[0]])
    ax.plot(*np.stack([probe - 2.2 * side, probe + 2.2 * side]).T, color="#1e40af", lw=7, solid_capstyle="butt")
    ax.plot(*np.stack([probe - 0.6 * beam, centre - (R_mm / max(beam[1], 0.3)) * beam]).T, "--", color="#f59e0b")
    ax.add_patch(Arc(centre, 6, 6, theta1=0, theta2=math.degrees(theta), color="#f59e0b", lw=1.5))
    ax.text(*(centre + 3.8 * np.array([math.cos(theta / 2), math.sin(theta / 2)])), "θ", color="#b45309")
    ax.text(probe[0] + 1.0, probe[1] + 1.2, f"{f0 / 1e6:.0f} MHz", fontsize=8, color="#1e40af")
    ax.annotate("", xy=(4.5, -R_mm - 0.9), xytext=(0.5, -R_mm - 0.9), arrowprops={"arrowstyle": "->", "lw": 1.2})
    ax.text(5.0, -R_mm - 1.3, "flow (shown 10× slower)", fontsize=7)
    vmax = float(v_max.abs().max())
    dots = ax.scatter(x_mm[0], y_mm, c=speed[0], s=9, cmap="plasma", vmin=0, vmax=vmax, zorder=3)
    cb = fig.colorbar(dots, ax=ax, shrink=0.7, pad=0.02)
    cb.set_label("speed (m/s)")
    title = ax.set_title("t = 0.00 s")

    # --- right: Doppler spectrogram (frequency or velocity vs time) -------------------------------------
    db = 10 * torch.log10(power / power.max() + 1e-12).clamp(min=-40).cpu().numpy()  # (F, K)
    wt = win_t.cpu().numpy()
    f_khz = freqs.cpu().numpy() / 1e3
    df = (f_khz[1] - f_khz[0]) / 2
    cmap = plt.get_cmap("magma", 16).with_extremes(bad="white")
    image = ax_s.imshow(
        np.full_like(db, np.nan),
        origin="lower",
        aspect="auto",
        cmap=cmap,
        vmin=-40,
        vmax=0,
        extent=[wt[0], wt[-1], f_khz[0] - df, f_khz[-1] + df],
        interpolation="nearest",
    )
    expected = doppler_shift(v_max, f0, theta).cpu().numpy() / 1e3
    ax_s.plot(times, expected, "--", color="#22d3ee", lw=1, label="$f_D$ of centreline $v_{\\max}(t)$")
    ax_s.set_ylim(-prf / 2e3, prf / 2e3)
    ax_s.set_xlim(0, times[-1])
    ax_s.set_xlabel("time (s)")
    ax_s.set_ylabel("Doppler shift (kHz)")
    ax_s.set_title(f"spectrogram, Nyquist ±PRF/2 = ±{prf / 2e3:.1f} kHz")
    ax_s.legend(loc="lower right", fontsize=7)
    ax_s.grid(False)
    if math.cos(theta) > 0.05:
        khz_per_ms = float(doppler_shift(torch.tensor(1.0), f0, theta)) / 1e3  # kHz of shift per m/s of flow
        velocity_axis = ax_s.secondary_yaxis("right", functions=(lambda f: f / khz_per_ms, lambda u: u * khz_per_ms))
        velocity_axis.set_ylabel("velocity (m/s)")
    cursor = ax_s.axvline(0, color="#9ca3af", lw=1)

    def update(k: int):
        step = frames[k]
        dots.set_offsets(np.column_stack([x_mm[k], y_mm]))
        dots.set_array(speed[k])
        shown = np.where(wt[None, :] <= times[step], db, np.nan)
        image.set_data(shown)
        cursor.set_xdata([times[step]])
        title.set_text(f"t = {times[step]:.2f} s   $v_{{\\max}}$ = {float(v_max[step]):.2f} m/s")

    fig.tight_layout()
    return save_gif(fig, update, list(range(len(frames))), path, fps=15)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'f0': 5.0,
    'theta': 60.0,
    'v_peak': 1.0,
    'prf': 10.0,
    'heart_rate': 72.0,
    'n_scatterers': 300
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.doppler_ultrasound")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.doppler_ultrasound.main")
problem_dir = Path("problems/medical/doppler_ultrasound")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
